# GIAI ĐOẠN 1: TRÍCH XUẤT ĐẶC TRƯNG VIDEO SANG PYTORCH TENSOR (.PT)
## NOTEBOOK: `extract_to_pt.ipynb` - TỐI ƯU HÓA SONG SONG ĐA GPU CUDA (MULTI-GPU PARALLEL)
---
### 1. Mục tiêu & Luồng hoạt động của Pipeline:
1. **Tự động quét & phát hiện thiết bị CUDA (CUDA Devices Auto-Scan):**
   * Tự động nhận diện tất cả GPU NVIDIA có sẵn trên hệ thống (`torch.cuda.device_count()`).
   * Hiển thị bảng thông số phần cứng chi tiết: Tên GPU, VRAM (GB), Compute Capability, số SMs.
   * Cho phép chạy song song trên toàn bộ các GPU khả dụng (ví dụ: Kaggle 2x Tesla T4, Server 4x RTX 4090/A100) hoặc chọn GPU tùy chỉnh.
2. **Phân chia dữ liệu phân tầng (Stratified Train/Val Split 80/20):**
   * Quét toàn bộ video từ thư mục dữ liệu (`.mp4`, `.avi`, `.mkv`, `.mov`,...).
   * Phân tách nhãn ở cấp độ Video ID: `0: Tỉnh táo (Driving / Alert)`, `1: Buồn ngủ (Drowsiness / Drowsy)`.
   * Lưu cấu hình vào `dataset_split.json` để **chống rò rỉ dữ liệu (No Data Leakage)** và đảm bảo tính tái lập 100%.
3. **Mô hình CNN PAFPN & ONNX Runtime CUDA Zero-Copy:**
   * Sử dụng mô hình `clone.onnx` với `CUDAExecutionProvider` và cơ chế **I/O Binding**.
   * Dữ liệu truyền trực tiếp giữa PyTorch CUDA Tensor và ONNX qua DLPack (Zero-copy), không qua RAM trung gian.
   * Kết hợp `AdaptiveAvgPool2d(1, 1)` trích xuất 3 tầng đặc trưng đa tỷ lệ:
     * $p_3$: $224$ chiều (kích thước gốc $60 \times 60$)
     * $p_4$: $448$ chiều (kích thước gốc $30 \times 30$)
     * $p_5$: $640$ chiều (kích thước gốc $15 \times 15$)
4. **Tăng cường dữ liệu (Data Augmentation) từ `augment.py`:**
   * Hỗ trợ tạo $N$ bản sao tăng cường ngẫu nhiên nhưng có seed xác định (Reproducible) cho tập Train.
5. **Kiến trúc Song song Đa GPU (Multi-GPU Parallel Dispatcher):**
   * Khởi tạo các tiến trình Worker độc lập gắn cố định với từng GPU (`cuda:0`, `cuda:1`,...).
   * Điều phối công việc động qua hàng đợi (`mp.Queue`), tự động cân bằng tải giữa các GPU.
   * Giám sát tiến độ trực quan thời gian thực bằng thanh tiến trình `tqdm` hợp nhất.
6. **Cơ chế Cache thông minh & Đóng gói Tensor (.pt):**
   * Lưu kết quả từng video độc lập vào thư mục `cache/` tạm thời (tự động khôi phục / resume nếu bị gián đoạn).
   * Đóng gói toàn bộ thành 2 tệp PyTorch Tensor nhị phân siêu tốc:
     * `features_sust_train.pt` (Tensor shape $[N_{train}, 120, C]$)
     * `features_sust_val.pt` (Tensor shape $[N_{val}, 120, C]$)


In [ ]:
# ==============================================================================
# CELL 1: THIẾT LẬP MÔI TRƯỜNG & NẠP THƯ VIỆN (ENVIRONMENT SETUP)
# ==============================================================================
import os
import sys
import time
import json
import random
import hashlib
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

# Đảm bảo console Windows/Linux hỗ trợ UTF-8 không bị lỗi mã hóa ký tự tiếng Việt
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

# Ngăn chặn xung đột OpenMP runtime trên Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Nạp thư viện CUDA DLL từ PyTorch nếu chạy trên môi trường Windows
import torch
torch_lib_path = os.path.join(os.path.dirname(torch.__file__), "lib")
if os.path.exists(torch_lib_path) and hasattr(os, "add_dll_directory"):
    try:
        os.add_dll_directory(torch_lib_path)
    except Exception as e:
        print(f"[WARN] Không thể thêm torch DLL directory: {e}")

import torch.nn.functional as F
import torch.multiprocessing as mp
import numpy as np
import cv2
from tqdm.auto import tqdm

try:
    import onnxruntime as ort
except ImportError as e:
    raise ImportError("Vui lòng cài đặt onnxruntime-gpu: pip install onnxruntime-gpu") from e

from augment import DetectionAugmenter, config as DEFAULT_AUG_CONFIG

print("=" * 75)
print(f"[+] Python Version       : {sys.version.split()[0]}")
print(f"[+] PyTorch Version      : {torch.__version__}")
print(f"[+] ONNX Runtime Version : {ort.__version__}")
print(f"[+] OpenCV Version       : {cv2.__version__}")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 2: TỰ ĐỘNG QUÉT & PHÁT HIỆN TẤT CẢ THIẾT BỊ CUDA (AUTO-SCAN CUDA DEVICES)
# ==============================================================================
def scan_cuda_devices() -> List[int]:
    """
    Quét toàn bộ thiết bị GPU CUDA có sẵn trên máy tính / môi trường điện toán.
    Hiển thị thông tin phần cứng chi tiết: Tên GPU, Dung lượng VRAM, Compute Capability.
    Trả về danh sách ID thiết bị khả dụng (ví dụ: [0] hoặc [0, 1]).
    """
    cuda_available = torch.cuda.is_available()
    ort_providers = ort.get_available_providers()
    cuda_ep_available = "CUDAExecutionProvider" in ort_providers

    print("=" * 80)
    print("           HỆ THỐNG QUÉT & KIỂM TRA PHẦN CỨNG TĂNG TỐC CUDA            ")
    print("=" * 80)
    print(f"[*] PyTorch CUDA Available       : {cuda_available}")
    print(f"[*] ONNX CUDAExecutionProvider   : {cuda_ep_available}")
    print(f"[*] ONNX Runtime Providers       : {ort_providers}")

    if not cuda_available:
        print("\n[!] CẢNH BÁO: Không tìm thấy GPU CUDA trên hệ thống.")
        print("    Các tác vụ sẽ chạy ở chế độ CPU (tốc độ sẽ rất chậm).")
        return []

    num_devices = torch.cuda.device_count()
    print(f"\n[+] TỔNG SỐ THIẾT BỊ CUDA PHÁT HIỆN: {num_devices}")
    print("-" * 80)
    print(f"{'Device ID':<12}{'Tên GPU':<32}{'VRAM (GB)':<14}{'Compute Cap':<14}{'SM Count':<10}")
    print("-" * 80)

    device_ids = []
    for dev_id in range(num_devices):
        props = torch.cuda.get_device_properties(dev_id)
        vram_gb = props.total_memory / (1024 ** 3)
        compute_cap = f"{props.major}.{props.minor}"
        sm_count = getattr(props, "multi_processor_count", "N/A")
        print(f"CUDA:{dev_id:<7}{props.name:<32}{vram_gb:<14.2f}{compute_cap:<14}{sm_count:<10}")
        device_ids.append(dev_id)

    print("-" * 80)
    if num_devices > 1:
        print(f"[+] HỆ THỐNG ĐA GPU: Sẵn sàng kích hoạt chạy song song trên {num_devices} GPUs: {device_ids}")
    else:
        print(f"[+] HỆ THỐNG ĐƠN GPU: Chạy trên GPU CUDA:{device_ids[0]}")
    print("=" * 80)
    return device_ids

# Quét và lưu danh sách GPU khả dụng
DETECTED_CUDA_DEVICES = scan_cuda_devices()


In [ ]:
# ==============================================================================
# CELL 3: CẤU HÌNH TRUNG TÂM (CENTRAL CONFIGURATION)
# ==============================================================================
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = "google.colab" in sys.modules

class ExtractionConfig:
    """Cấu hình toàn bộ tham số cho tiến trình trích xuất đặc trưng đa GPU."""

    # 1. ĐƯỜNG DẪN DỮ LIỆU & MÔ HÌNH
    if IS_KAGGLE:
        data_dir = Path("/kaggle/input/sust-dataset/SUST")
        onnx_path = Path("/kaggle/input/sust-models/clone.onnx")
        output_dir = Path("/kaggle/working/extracted_features_pt")
    elif IS_COLAB:
        data_dir = Path("/content/dataset/SUST")
        onnx_path = Path("/content/clone.onnx")
        output_dir = Path("/content/extracted_features_pt")
    else:
        # Đường dẫn cục bộ (Local PC)
        data_dir = Path(r"D:\Project\AI\dataset\SUST")
        onnx_path = Path(r"outsrc\myCNN\checkpoints_ftCOCO\clone.onnx")
        output_dir = Path("extracted_features_pt")

    video_exts: Tuple[str, ...] = (".mp4", ".avi", ".mkv", ".mov")
    train_name: str = "features_sust_train.pt"
    val_name: str = "features_sust_val.pt"
    cache_dir: Path = output_dir / "cache"
    split_json_path: Path = output_dir / "dataset_split.json"

    # 2. THAM SỐ XỬ LÝ KHUNG HÌNH & TRÍCH XUẤT
    seq_len: int = 120              # Số khung hình chuẩn hóa mỗi video
    img_size: int = 480             # Kích thước resize letterbox (480x480)
    chunk_size: int = 24            # Kích thước mini-chunk forward ONNX (120 / 24 = 5 chunks)
    use_fp16: bool = False          # Lưu tensor dạng float16 (tiết kiệm 50% dung lượng)

    # 3. TĂNG CƯỜNG DỮ LIỆU (DATA AUGMENTATION)
    augment: bool = True            # Bật/tắt tăng cường dữ liệu
    num_aug: int = 1                # Số bản sao tăng cường cho mỗi video
    include_original: bool = True   # Giữ lại video gốc bên cạnh bản augment
    augment_splits: Tuple[str, ...] = ("train",)  # Chỉ áp dụng augment cho tập train
    aug_seed: int = 42              # Seed cơ sở cho quá trình augment

    # 4. CẤU HÌNH ĐA THIẾT BỊ GPU CUDA (MULTI-GPU PARALLELISM)
    # Mặc định sử dụng toàn bộ GPU CUDA phát hiện được. Có thể cấu hình tùy chỉnh ví dụ [0, 1]
    device_ids: List[int] = DETECTED_CUDA_DEVICES if DETECTED_CUDA_DEVICES else [0]
    
    # 5. ĐIỀU KHIỂN QUÁ TRÌNH
    force_recompute: bool = False   # Bắt buộc tính toán lại, bỏ qua cache
    limit: Optional[int] = None     # Giới hạn số lượng video để chạy thử nghiệm (None = toàn bộ)

cfg = ExtractionConfig()

print("=" * 80)
print("                    CẤU HÌNH TIẾN TRÌNH TRÍCH XUẤT                    ")
print("=" * 80)
print(f"[*] Dataset Dir       : {cfg.data_dir} (Tồn tại: {cfg.data_dir.exists()})")
print(f"[*] Video Exts        : {cfg.video_exts}")
print(f"[*] ONNX Model Path   : {cfg.onnx_path} (Tồn tại: {cfg.onnx_path.exists()})")
print(f"[*] Output Dir        : {cfg.output_dir}")
print(f"[*] Cache Dir         : {cfg.cache_dir}")
print(f"[*] Split JSON        : {cfg.split_json_path}")
print(f"[*] Sequence Length   : {cfg.seq_len} frames")
print(f"[*] Image Size        : {cfg.img_size}x{cfg.img_size}")
print(f"[*] Chunk Size        : {cfg.chunk_size}")
print(f"[*] Precision         : {'Float16' if cfg.use_fp16 else 'Float32'}")
print(f"[*] Data Augmentation : {'BẬT' if cfg.augment else 'TẮT'} (num_aug={cfg.num_aug}, splits={cfg.augment_splits})")
print(f"[*] CUDA Devices      : {cfg.device_ids} ({len(cfg.device_ids)} GPUs song song)")
print(f"[*] Force Recompute   : {cfg.force_recompute}")
if cfg.limit:
    print(f"[*] TEST MODE (Limit) : {cfg.limit} videos")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: TIỀN XỬ LÝ ẢNH & RUNTIME ONNX CUDA ZERO-COPY (I/O BINDING)
# ==============================================================================
def letterbox(image: np.ndarray, new_size: int = 480, color=(114, 114, 114)) -> np.ndarray:
    """Resize ảnh giữ nguyên tỉ lệ (aspect ratio) với padding đồng màu."""
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    pad_left = (new_size - new_w) // 2
    pad_top = (new_size - new_h) // 2
    canvas[pad_top: pad_top + new_h, pad_left: pad_left + new_w] = resized
    return canvas


class CloneONNXCUDARuntime:
    """Quản lý suy luận ONNX Runtime trên GPU CUDA với I/O Binding Zero-Copy."""

    def __init__(self, onnx_model_path: str, device_id: int = 0):
        self.onnx_model_path = str(onnx_model_path)
        self.device_id = device_id

        if not os.path.exists(self.onnx_model_path):
            raise FileNotFoundError(f"Không tìm thấy file ONNX: {self.onnx_model_path}")

        self.session_options = ort.SessionOptions()
        self.session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session_options.log_severity_level = 3

        cuda_options = {
            "device_id": self.device_id,
            "arena_extend_strategy": "kNextPowerOfTwo",
            "cudnn_conv_algo_search": "HEURISTIC",
            "do_copy_in_default_stream": True,
        }
        providers = [
            ("CUDAExecutionProvider", cuda_options),
            "CPUExecutionProvider"
        ]

        self.session = ort.InferenceSession(
            self.onnx_model_path,
            sess_options=self.session_options,
            providers=providers,
        )

        active = self.session.get_providers()
        if "CUDAExecutionProvider" not in active:
            raise RuntimeError(f"CUDAExecutionProvider chưa được kích hoạt trên GPU {self.device_id}! Providers: {active}")

        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]

    def forward(self, input_tensor: torch.Tensor) -> List[torch.Tensor]:
        """Suy luận trực tiếp từ PyTorch CUDA Tensor sang PyTorch CUDA Tensors (Zero-copy DLPack)."""
        io_binding = self.session.io_binding()
        io_binding.bind_input(
            name=self.input_name,
            device_type="cuda",
            device_id=self.device_id,
            element_type=np.float32,
            shape=tuple(input_tensor.shape),
            buffer_ptr=input_tensor.data_ptr(),
        )
        for out_name in self.output_names:
            io_binding.bind_output(out_name, device_type="cuda", device_id=self.device_id)

        self.session.run_with_iobinding(io_binding)
        raw_outputs = io_binding.get_outputs()
        return [torch.from_dlpack(out) for out in raw_outputs]


def read_and_sample_video_frames(
        video_path: Path,
        seq_len: int = 120,
        img_size: int = 480
) -> Tuple[List[np.ndarray], int]:
    """
    Đọc nhanh video tuần tự, letterbox sang kích thước cố định và định dạng RGB (uint8).
    Hỗ trợ đa định dạng video (.mp4, .avi, .mkv, .mov,...).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Không thể mở video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []

    if total_frames <= 0:
        raw_frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            raw_frames.append(frame)
        cap.release()

        n_raw = len(raw_frames)
        if n_raw > 0:
            indices = set(torch.linspace(0, max(0, n_raw - 1), seq_len).long().tolist())
            for idx, frame in enumerate(raw_frames):
                if idx in indices:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    lb = letterbox(rgb, new_size=img_size)
                    frames.append(lb)
    else:
        indices = set(torch.linspace(0, max(0, total_frames - 1), seq_len).long().tolist())
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx in indices:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                lb = letterbox(rgb, new_size=img_size)
                frames.append(lb)
            idx += 1
        cap.release()

    actual_frames = len(frames)
    if actual_frames == 0:
        frames = [np.zeros((img_size, img_size, 3), dtype=np.uint8) for _ in range(seq_len)]
    else:
        while len(frames) < seq_len:
            frames.append(frames[-1].copy())
    frames = frames[:seq_len]
    return frames, actual_frames


def forward_video_chunks(
        frames_rgb: List[np.ndarray],
        runner: CloneONNXCUDARuntime,
        chunk_size: int = 24,
        device: str = "cuda:0"
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Forward danh sách khung hình RGB qua mô hình ONNX CUDA theo từng mini-chunk
    kết hợp Adaptive Average Pooling (1, 1).
    """
    seq_len = len(frames_rgb)
    tensor_list = [
        torch.from_numpy(np.ascontiguousarray(f.transpose(2, 0, 1))).float() / 255.0
        for f in frames_rgb
    ]
    video_tensor = torch.stack(tensor_list, dim=0)

    p3_chunks, p4_chunks, p5_chunks = [], [], []
    for i in range(0, seq_len, chunk_size):
        chunk = video_tensor[i: i + chunk_size].to(device, non_blocking=True)
        outs = runner.forward(chunk)
        p3_chunks.append(F.adaptive_avg_pool2d(outs[0], (1, 1)).flatten(1).cpu())
        p4_chunks.append(F.adaptive_avg_pool2d(outs[1], (1, 1)).flatten(1).cpu())
        p5_chunks.append(F.adaptive_avg_pool2d(outs[2], (1, 1)).flatten(1).cpu())

    p3_tensor = torch.cat(p3_chunks, dim=0)  # [seq_len, 224]
    p4_tensor = torch.cat(p4_chunks, dim=0)  # [seq_len, 448]
    p5_tensor = torch.cat(p5_chunks, dim=0)  # [seq_len, 640]
    return p3_tensor, p4_tensor, p5_tensor


In [ ]:
# ==============================================================================
# CELL 5: QUẢN LÝ PHÂN CHIA DATASET PHÂN TẦNG (STRATIFIED DATASET PRE-SPLITTING)
# ==============================================================================
def prepare_dataset_split(
        dataset_dir: Path,
        split_json_path: Path,
        train_ratio: float = 0.8,
        seed: int = 42,
        video_exts: Tuple[str, ...] = (".mp4", ".avi", ".mkv", ".mov")
) -> Tuple[List[Tuple[Path, int]], List[Tuple[Path, int]]]:
    """
    Quét video, phân chia Train/Val theo Stratified Split ở cấp độ Video ID trước khi trích xuất.
    Nếu file dataset_split.json đã tồn tại và hợp lệ, sẽ tái sử dụng để bảo đảm tính nhất quán.
    """
    if split_json_path.exists():
        print(f"[+] Tìm thấy file cấu hình phân chia có sẵn: {split_json_path}")
        with open(split_json_path, "r", encoding="utf-8") as f:
            split_info = json.load(f)

        train_items = [(dataset_dir / item["name"], item["label"]) for item in split_info.get("train", [])]
        val_items = [(dataset_dir / item["name"], item["label"]) for item in split_info.get("val", [])]

        all_items = train_items + val_items
        if all_items and all(p.exists() for p, _ in all_items[:10]):
            print(f"[+] Đã tải phân chia thành công: Train={len(train_items)} video, Val={len(val_items)} video.")
            return train_items, val_items
        else:
            print(f"[WARN] File '{split_json_path}' không khớp với video trong '{dataset_dir}'. Đang phân chia lại...")

    print(f"[+] Quét toàn bộ video từ thư mục: {dataset_dir}...")
    normalized_exts = {ext.lower() if ext.startswith(".") else f".{ext.lower()}" for ext in video_exts}

    video_files = [
        f for f in dataset_dir.iterdir()
        if f.is_file() and f.suffix.lower() in normalized_exts
    ]
    if not video_files:
        video_files = [
            f for f in dataset_dir.rglob("*")
            if f.is_file() and f.suffix.lower() in normalized_exts
        ]
    video_files = sorted(video_files)

    if not video_files:
        raise FileNotFoundError(
            f"Không tìm thấy video nào thuộc các định dạng {sorted(list(normalized_exts))} trong: {dataset_dir}"
        )

    items = []
    for vf in video_files:
        stem = vf.stem
        parts = stem.split("-")
        label = None
        if len(parts) >= 2:
            label_str = parts[-2].lower()
            if label_str in ("driving", "alert", "normal"):
                label = 0
            elif label_str in ("drowsiness", "drowsy"):
                label = 1

        if label is None:
            name_lower = stem.lower()
            if "drowsiness" in name_lower or "drowsy" in name_lower:
                label = 1
            elif "driving" in name_lower or "alert" in name_lower or "normal" in name_lower:
                label = 0

        if label is not None:
            items.append((vf, label))

    print(f"[+] Tìm thấy tổng cộng {len(items)} video hợp lệ có nhãn.")
    if not items:
        raise ValueError(f"Không thể trích xuất nhãn 'driving' hoặc 'drowsiness' từ video trong: {dataset_dir}")

    # Phân chia phân tầng (Stratified Split)
    rng = random.Random(seed)
    class_0 = [item for item in items if item[1] == 0]
    class_1 = [item for item in items if item[1] == 1]

    rng.shuffle(class_0)
    rng.shuffle(class_1)

    split_0 = int(round(len(class_0) * train_ratio))
    split_1 = int(round(len(class_1) * train_ratio))

    train_items = class_0[:split_0] + class_1[:split_1]
    val_items = class_0[split_0:] + class_1[split_1:]

    rng.shuffle(train_items)
    rng.shuffle(val_items)

    # Lưu siêu dữ liệu phân bổ
    split_info = {
        "metadata": {
            "dataset_dir": str(dataset_dir),
            "seed": seed,
            "train_ratio": train_ratio,
            "total_videos": len(items),
            "train_count": len(train_items),
            "val_count": len(val_items),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "train": [
            {"name": p.relative_to(dataset_dir).as_posix() if p.is_relative_to(dataset_dir) else p.name, "label": lbl}
            for p, lbl in train_items],
        "val": [
            {"name": p.relative_to(dataset_dir).as_posix() if p.is_relative_to(dataset_dir) else p.name, "label": lbl}
            for p, lbl in val_items]
    }

    split_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(split_json_path, "w", encoding="utf-8") as f:
        json.dump(split_info, f, indent=4, ensure_ascii=False)

    print(f"[+] Đã lưu cấu hình phân bổ độc lập tại: {split_json_path}")
    print(f"    - Tập Train: {len(train_items)} videos ({len(train_items) / len(items) * 100:.1f}%)")
    print(f"    - Tập Val  : {len(val_items)} videos ({len(val_items) / len(items) * 100:.1f}%)")

    return train_items, val_items


In [ ]:
# ==============================================================================
# CELL 6: KIẾN TRÚC WORKER ĐỘC LẬP CHO TIẾN TRÌNH CON MULTIPROCESSING
# ==============================================================================
# Đảm bảo tệp multi_gpu_worker.py sẵn sàng để các tiến trình con (spawns) có thể import
# mà không gặp lỗi AttributeError / PicklingError trên Windows và Linux/Kaggle.
worker_file_path = Path("multi_gpu_worker.py")
if not worker_file_path.exists():
    print(f"[+] Tạo tệp worker module song song: {worker_file_path.resolve()}...")
    worker_module_content = r'''import os, sys, time, hashlib
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"): sys.stderr.reconfigure(encoding="utf-8")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn.functional as F
import numpy as np
import cv2

torch_lib_path = os.path.join(os.path.dirname(torch.__file__), "lib")
if os.path.exists(torch_lib_path) and hasattr(os, "add_dll_directory"):
    try: os.add_dll_directory(torch_lib_path)
    except Exception: pass

import onnxruntime as ort
from augment import DetectionAugmenter, config as DEFAULT_AUG_CONFIG

def letterbox(image: np.ndarray, new_size: int = 480, color=(114, 114, 114)) -> np.ndarray:
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    pad_left = (new_size - new_w) // 2
    pad_top = (new_size - new_h) // 2
    canvas[pad_top: pad_top + new_h, pad_left: pad_left + new_w] = resized
    return canvas

class CloneONNXCUDARuntime:
    def __init__(self, onnx_model_path: str, device_id: int = 0):
        self.onnx_model_path = str(onnx_model_path)
        self.device_id = device_id
        if not os.path.exists(self.onnx_model_path):
            raise FileNotFoundError(f"Không tìm thấy ONNX: {self.onnx_model_path}")
        self.session_options = ort.SessionOptions()
        self.session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session_options.log_severity_level = 3
        cuda_options = {
            "device_id": self.device_id,
            "arena_extend_strategy": "kNextPowerOfTwo",
            "cudnn_conv_algo_search": "HEURISTIC",
            "do_copy_in_default_stream": True,
        }
        providers = [("CUDAExecutionProvider", cuda_options), "CPUExecutionProvider"]
        self.session = ort.InferenceSession(self.onnx_model_path, sess_options=self.session_options, providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]

    def forward(self, input_tensor: torch.Tensor) -> List[torch.Tensor]:
        io_binding = self.session.io_binding()
        io_binding.bind_input(
            name=self.input_name, device_type="cuda", device_id=self.device_id,
            element_type=np.float32, shape=tuple(input_tensor.shape), buffer_ptr=input_tensor.data_ptr(),
        )
        for out_name in self.output_names:
            io_binding.bind_output(out_name, device_type="cuda", device_id=self.device_id)
        self.session.run_with_iobinding(io_binding)
        raw_outputs = io_binding.get_outputs()
        return [torch.from_dlpack(out) for out in raw_outputs]

def read_and_sample_video_frames(video_path: Path, seq_len: int = 120, img_size: int = 480):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Không thể mở video: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total_frames <= 0:
        raw_frames = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            raw_frames.append(frame)
        cap.release()
        n_raw = len(raw_frames)
        if n_raw > 0:
            indices = set(torch.linspace(0, max(0, n_raw - 1), seq_len).long().tolist())
            for idx, frame in enumerate(raw_frames):
                if idx in indices:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(letterbox(rgb, new_size=img_size))
    else:
        indices = set(torch.linspace(0, max(0, total_frames - 1), seq_len).long().tolist())
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            if idx in indices:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(letterbox(rgb, new_size=img_size))
            idx += 1
        cap.release()
    actual_frames = len(frames)
    if actual_frames == 0:
        frames = [np.zeros((img_size, img_size, 3), dtype=np.uint8) for _ in range(seq_len)]
    else:
        while len(frames) < seq_len:
            frames.append(frames[-1].copy())
    return frames[:seq_len], actual_frames

def forward_video_chunks(frames_rgb, runner, chunk_size=24, device="cuda:0"):
    seq_len = len(frames_rgb)
    tensor_list = [torch.from_numpy(np.ascontiguousarray(f.transpose(2, 0, 1))).float() / 255.0 for f in frames_rgb]
    video_tensor = torch.stack(tensor_list, dim=0)
    p3_chunks, p4_chunks, p5_chunks = [], [], []
    for i in range(0, seq_len, chunk_size):
        chunk = video_tensor[i: i + chunk_size].to(device, non_blocking=True)
        outs = runner.forward(chunk)
        p3_chunks.append(F.adaptive_avg_pool2d(outs[0], (1, 1)).flatten(1).cpu())
        p4_chunks.append(F.adaptive_avg_pool2d(outs[1], (1, 1)).flatten(1).cpu())
        p5_chunks.append(F.adaptive_avg_pool2d(outs[2], (1, 1)).flatten(1).cpu())
    return torch.cat(p3_chunks, dim=0), torch.cat(p4_chunks, dim=0), torch.cat(p5_chunks, dim=0)

def gpu_worker_loop(worker_id, gpu_id, task_queue, result_queue, onnx_model_path, split_cache_dir_str,
                    seq_len=120, img_size=480, chunk_size=24, augment_enabled=False, aug_cfg=None,
                    num_aug=0, include_original=True, base_seed=42, force_recompute=False):
    try:
        torch.cuda.set_device(gpu_id)
        device_str = f"cuda:{gpu_id}"
        runner = CloneONNXCUDARuntime(onnx_model_path=onnx_model_path, device_id=gpu_id)
        dummy = torch.zeros(chunk_size, 3, img_size, img_size, device=device_str)
        runner.forward(dummy)
        del dummy
        torch.cuda.empty_cache()

        augmenter = None
        if augment_enabled and num_aug > 0:
            cfg = dict(DEFAULT_AUG_CONFIG)
            if aug_cfg: cfg.update(aug_cfg)
            augmenter = DetectionAugmenter(cfg)

        split_cache_dir = Path(split_cache_dir_str)
        split_cache_dir.mkdir(parents=True, exist_ok=True)
        is_augmented = (augmenter is not None and num_aug > 0)
        actual_include_original = include_original if is_augmented else True

        result_queue.put({"type": "ready", "worker_id": worker_id, "gpu_id": gpu_id})

        while True:
            task = task_queue.get()
            if task is None: break
            video_path_str, label = task
            video_path = Path(video_path_str)
            video_id = video_path.stem

            targets = []
            if actual_include_original:
                targets.append({"sub_id": video_id, "is_aug": False, "aug_idx": 0, "seed": None})
            if is_augmented:
                for k in range(1, num_aug + 1):
                    aug_id = f"{video_id}_aug{k}"
                    aug_seed = (base_seed + int(hashlib.md5(aug_id.encode("utf-8")).hexdigest()[:8], 16)) % (2 ** 31 - 1)
                    targets.append({"sub_id": aug_id, "is_aug": True, "aug_idx": k, "seed": aug_seed})

            needed = []
            for t in targets:
                cache_file = split_cache_dir / f"{t['sub_id']}.pt"
                if force_recompute or not cache_file.exists():
                    needed.append(t)

            if len(needed) == 0:
                result_queue.put({"type": "done", "worker_id": worker_id, "gpu_id": gpu_id, "video_id": video_id, "status": "cached", "num_targets": len(targets)})
                continue

            try:
                raw_frames, actual_frames = read_and_sample_video_frames(video_path, seq_len=seq_len, img_size=img_size)
                for t in needed:
                    if t["is_aug"]:
                        aug_frames, _, _, _ = augmenter.augment_video(frames=raw_frames, boxes_list=None, labels_list=None, seed=t["seed"])
                        p3, p4, p5 = forward_video_chunks(aug_frames, runner, chunk_size=chunk_size, device=device_str)
                    else:
                        p3, p4, p5 = forward_video_chunks(raw_frames, runner, chunk_size=chunk_size, device=device_str)

                    torch.save({"video_id": t["sub_id"], "label": label, "p3": p3, "p4": p4, "p5": p5,
                                "actual_frames": actual_frames, "is_augmented": t["is_aug"], "aug_seed": t["seed"]},
                                split_cache_dir / f"{t['sub_id']}.pt")

                result_queue.put({"type": "done", "worker_id": worker_id, "gpu_id": gpu_id, "video_id": video_id, "status": "extracted", "num_targets": len(targets)})
            except Exception as e:
                result_queue.put({"type": "error", "worker_id": worker_id, "gpu_id": gpu_id, "video_id": video_id, "error": str(e)})

    except Exception as fatal_e:
        result_queue.put({"type": "fatal_error", "worker_id": worker_id, "gpu_id": gpu_id, "error": str(fatal_e)})
'''
    with open(worker_file_path, "w", encoding="utf-8") as f:
        f.write(worker_module_content)

import multi_gpu_worker
print(f"[+] Đã sẵn sàng module worker song song: {multi_gpu_worker.__file__}")


In [ ]:
# ==============================================================================
# CELL 7: BỘ ĐIỀU PHỐI TRÍCH XUẤT SONG SONG ĐA GPU (MULTI-GPU PARALLEL ENGINE)
# ==============================================================================
def run_multi_gpu_extraction(
        split_name: str,
        items: List[Tuple[Path, int]],
        cfg: ExtractionConfig,
        augment_this_split: bool = False
) -> None:
    """
    Điều phối trích xuất đặc trưng song song trên tất cả các thiết bị CUDA được cấu hình.
    Sử dụng hàng đợi công việc linh hoạt (Dynamic Task Queue) và theo dõi tiến độ qua tqdm.
    """
    device_ids = cfg.device_ids
    num_workers = len(device_ids)
    split_cache_dir = cfg.cache_dir / split_name
    split_cache_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"[*] TIẾN TRÌNH TRÍCH XUẤT ĐA GPU PHÂN TẬP: {split_name.upper()} ({len(items)} VIDEOS)")
    print(f"[*] Số lượng thiết bị GPU tham gia: {num_workers} ({['CUDA:' + str(i) for i in device_ids]})")
    print(f"[*] Tăng cường dữ liệu (Augmentation): {'BẬT (num_aug=' + str(cfg.num_aug) + ')' if augment_this_split else 'TẮT'}")
    print(f"[*] Thư mục Cache: {split_cache_dir}")
    print("=" * 80)

    # Nếu chỉ có 1 GPU và muốn chạy trực tiếp không spawn multiprocessing
    if num_workers == 1:
        print(f"[+] Chạy chế độ Single-GPU tối ưu trên CUDA:{device_ids[0]}...")
        ctx = mp.get_context("spawn")
    else:
        print(f"[+] Khởi tạo Multi-GPU Worker Pool trên {num_workers} GPUs...")
        ctx = mp.get_context("spawn")

    task_queue = ctx.Queue()
    result_queue = ctx.Queue()

    # Nạp toàn bộ video vào task queue
    for video_path, label in items:
        task_queue.put((str(video_path), label))
    for _ in range(num_workers):
        task_queue.put(None)  # Tín hiệu kết thúc cho mỗi worker

    workers = []
    for worker_id, gpu_id in enumerate(device_ids):
        p = ctx.Process(
            target=multi_gpu_worker.gpu_worker_loop,
            kwargs=dict(
                worker_id=worker_id,
                gpu_id=gpu_id,
                task_queue=task_queue,
                result_queue=result_queue,
                onnx_model_path=str(cfg.onnx_path),
                split_cache_dir_str=str(split_cache_dir),
                seq_len=cfg.seq_len,
                img_size=cfg.img_size,
                chunk_size=cfg.chunk_size,
                augment_enabled=augment_this_split,
                aug_cfg=None,
                num_aug=cfg.num_aug if augment_this_split else 0,
                include_original=cfg.include_original,
                base_seed=cfg.aug_seed,
                force_recompute=cfg.force_recompute
            )
        )
        p.start()
        workers.append(p)

    # Chờ tất cả worker gửi tín hiệu ready
    ready_count = 0
    while ready_count < num_workers:
        msg = result_queue.get()
        if msg.get("type") == "ready":
            ready_count += 1
            print(f"    -> Worker {msg['worker_id']} sẵn sàng trên GPU CUDA:{msg['gpu_id']}")
        elif msg.get("type") == "fatal_error":
            print(f"[FATAL ERROR] Worker {msg['worker_id']} trên GPU CUDA:{msg['gpu_id']} gặp lỗi: {msg['error']}")
            for p in workers:
                p.terminate()
            raise RuntimeError(f"Worker khởi động thất bại: {msg['error']}")

    print(f"[+] Tất cả {num_workers} GPU Workers đã sẵn sàng! Bắt đầu trích xuất song song...")

    # Theo dõi tiến độ qua thanh tqdm
    pbar = tqdm(total=len(items), desc=f"[{split_name.upper()} - {num_workers} GPUs]")
    cached_count = 0
    extracted_count = 0
    error_count = 0
    completed_videos = 0

    while completed_videos < len(items):
        msg = result_queue.get()
        msg_type = msg.get("type")

        if msg_type == "done":
            completed_videos += 1
            status = msg.get("status")
            if status == "cached":
                cached_count += 1
            else:
                extracted_count += 1
            pbar.set_postfix({
                "GPU": f"CUDA:{msg.get('gpu_id')}",
                "Mới": extracted_count,
                "Cache": cached_count,
                "Lỗi": error_count
            })
            pbar.update(1)

        elif msg_type == "error":
            completed_videos += 1
            error_count += 1
            print(f"\n[WARN] Lỗi khi xử lý video {msg.get('video_id')} trên GPU CUDA:{msg.get('gpu_id')}: {msg.get('error')}")
            pbar.update(1)

        elif msg_type == "fatal_error":
            print(f"\n[FATAL] Lỗi nghiêm trọng tại worker {msg.get('worker_id')} (GPU CUDA:{msg.get('gpu_id')}): {msg.get('error')}")
            break

    pbar.close()

    # Chờ tất cả tiến trình hoàn tất
    for p in workers:
        p.join()

    print(f"[+] Hoàn tất xử lý tập {split_name.upper()}: Tổng={completed_videos} (Mới={extracted_count}, Cache={cached_count}, Lỗi={error_count})")


In [ ]:
# ==============================================================================
# CELL 8: ĐÓNG GÓI PYTORCH TENSORS (.PT) CHO MỖI TẬP DỮ LIỆU
# ==============================================================================
def package_split_dataset(
        split_name: str,
        items: List[Tuple[Path, int]],
        output_pt_path: Path,
        cache_dir: Path,
        cfg: ExtractionConfig,
        augment_this_split: bool = False
) -> None:
    """
    Đọc tất cả các tệp đặc trưng .pt từ thư mục cache, hợp nhất thành tensor 3D
    và lưu thành tệp PyTorch nhị phân cuối cùng (features_sust_train.pt / val.pt).
    """
    split_cache_dir = cache_dir / split_name
    print(f"\n[+] Đang đọc dữ liệu từ cache: {split_cache_dir}...")

    is_augmented_split = (cfg.augment and augment_this_split and cfg.num_aug > 0)
    actual_include_original = cfg.include_original if is_augmented_split else True

    p3_all, p4_all, p5_all = [], [], []
    labels_all, video_ids_all, seq_lens_all = [], [], []

    missing_samples = 0
    for video_path, label in items:
        video_id = video_path.stem
        targets = []
        if actual_include_original:
            targets.append(video_id)
        if is_augmented_split:
            for k in range(1, cfg.num_aug + 1):
                targets.append(f"{video_id}_aug{k}")

        for sub_id in targets:
            cache_file = split_cache_dir / f"{sub_id}.pt"
            if cache_file.exists():
                cached_data = torch.load(cache_file, map_location="cpu")
                p3_all.append(cached_data["p3"])
                p4_all.append(cached_data["p4"])
                p5_all.append(cached_data["p5"])
                labels_all.append(cached_data["label"])
                video_ids_all.append(cached_data["video_id"])
                seq_lens_all.append(cached_data.get("actual_frames", cfg.seq_len))
            else:
                missing_samples += 1

    if missing_samples > 0:
        print(f"[WARN] Có {missing_samples} mẫu chưa được tạo trong cache!")

    if len(video_ids_all) == 0:
        raise RuntimeError(f"Không tìm thấy mẫu nào hợp lệ trong cache cho tập {split_name}!")

    print(f"[+] Đang đóng gói {len(video_ids_all)} mẫu vào tensor 3D...")
    final_p3 = torch.stack(p3_all, dim=0)  # [N, 120, 224]
    final_p4 = torch.stack(p4_all, dim=0)  # [N, 120, 448]
    final_p5 = torch.stack(p5_all, dim=0)  # [N, 120, 640]
    final_labels = torch.tensor(labels_all, dtype=torch.long)
    final_seq_lens = torch.tensor(seq_lens_all, dtype=torch.int32)

    if cfg.use_fp16:
        print("[+] Ép kiểu sang Float16 (giảm 50% dung lượng tệp)...\n")
        final_p3 = final_p3.half()
        final_p4 = final_p4.half()
        final_p5 = final_p5.half()

    num_alerts = (final_labels == 0).sum().item()
    num_drowsy = (final_labels == 1).sum().item()
    num_orig = sum(1 for vid in video_ids_all if "_aug" not in vid)
    num_aug_count = sum(1 for vid in video_ids_all if "_aug" in vid)

    save_dict = {
        "p3": final_p3,
        "p4": final_p4,
        "p5": final_p5,
        "labels": final_labels,
        "video_ids": video_ids_all,
        "seq_lens": final_seq_lens,
        "dtype": "float16" if cfg.use_fp16 else "float32",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "augmented": is_augmented_split,
        "num_aug": cfg.num_aug if is_augmented_split else 0,
        "include_original": actual_include_original,
    }

    output_pt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(save_dict, str(output_pt_path))

    file_size_mb = output_pt_path.stat().st_size / (1024 * 1024)
    print("=" * 80)
    print(f"[SUCCESS] ĐÃ LƯU THÀNH CÔNG TẬP {split_name.upper()} VÀO: {output_pt_path.resolve()}")
    print("=" * 80)
    print(f"  * Kích thước tệp       : {file_size_mb:.2f} MB")
    print(f"  * Tổng số mẫu tensor   : {len(video_ids_all)} (Gốc: {num_orig}, Augment: {num_aug_count})")
    print(f"  * Phân bố nhãn         : Tỉnh táo (0) = {num_alerts}, Buồn ngủ (1) = {num_drowsy}")
    print(f"  * Tensor p3 shape      : {list(final_p3.shape)}")
    print(f"  * Tensor p4 shape      : {list(final_p4.shape)}")
    print(f"  * Tensor p5 shape      : {list(final_p5.shape)}")
    print(f"  * Labels shape         : {list(final_labels.shape)}")
    print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 9: THỰC THI TOÀN BỘ QUY TRÌNH TRÍCH XUẤT ĐA GPU (PIPELINE EXECUTION)
# ==============================================================================
t_start_all = time.time()

# 1. Quét và phân chia Train / Validation
train_items, val_items = prepare_dataset_split(
    dataset_dir=cfg.data_dir,
    split_json_path=cfg.split_json_path,
    train_ratio=0.8,
    seed=42,
    video_exts=cfg.video_exts
)

# Áp dụng giới hạn nếu chạy thử nghiệm (Limit Mode)
if cfg.limit:
    train_limit = max(1, int(cfg.limit * 0.8))
    val_limit = max(1, cfg.limit - train_limit)
    train_items = train_items[:train_limit]
    val_items = val_items[:val_limit]
    print(f"[*] ÁP DỤNG GIỚI HẠN TEST: Train={len(train_items)} videos, Val={len(val_items)} videos.")

# 2. Trích xuất song song tập TRAIN
train_pt_path = cfg.output_dir / cfg.train_name
should_aug_train = cfg.augment and ("train" in cfg.augment_splits)
run_multi_gpu_extraction(
    split_name="train",
    items=train_items,
    cfg=cfg,
    augment_this_split=should_aug_train
)
package_split_dataset(
    split_name="train",
    items=train_items,
    output_pt_path=train_pt_path,
    cache_dir=cfg.cache_dir,
    cfg=cfg,
    augment_this_split=should_aug_train
)

# 3. Trích xuất song song tập VALIDATION
val_pt_path = cfg.output_dir / cfg.val_name
should_aug_val = cfg.augment and ("val" in cfg.augment_splits)
run_multi_gpu_extraction(
    split_name="val",
    items=val_items,
    cfg=cfg,
    augment_this_split=should_aug_val
)
package_split_dataset(
    split_name="val",
    items=val_items,
    output_pt_path=val_pt_path,
    cache_dir=cfg.cache_dir,
    cfg=cfg,
    augment_this_split=should_aug_val
)

total_elapsed_min = (time.time() - t_start_all) / 60
print("\n" + "=" * 80)
print(f"      [HOÀN TẤT GIAI ĐOẠN 1] TRÍCH XUẤT ĐẶC TRƯNG ĐA GPU THÀNH CÔNG!     ")
print("=" * 80)
print(f"1. Thời gian thực thi tổng cộng : {total_elapsed_min:.2f} phút")
print(f"2. Cấu hình phân chia dataset   : {cfg.split_json_path.resolve()}")
print(f"3. File Tensor Train (.pt)       : {train_pt_path.resolve()}")
print(f"4. File Tensor Val (.pt)         : {val_pt_path.resolve()}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 10: KIỂM TRA TÍNH HỢP LỆ CỦA TENSOR & TRỰC QUAN HÓA (VERIFICATION)
# ==============================================================================
import matplotlib.pyplot as plt

print("=" * 80)
print("             KIỂM TRA DỮ LIỆU TENSOR (.PT) VỪA TRÍCH XUẤT             ")
print("=" * 80)

# Nạp kiểm tra 2 file .pt
train_data = torch.load(str(train_pt_path), map_location="cpu")
val_data = torch.load(str(val_pt_path), map_location="cpu")

print(f"[+] TRAIN TENSOR:")
print(f"    - p3: {list(train_data['p3'].shape)} | dtype: {train_data['p3'].dtype} | NaNs: {torch.isnan(train_data['p3']).any().item()}")
print(f"    - p4: {list(train_data['p4'].shape)} | dtype: {train_data['p4'].dtype} | NaNs: {torch.isnan(train_data['p4']).any().item()}")
print(f"    - p5: {list(train_data['p5'].shape)} | dtype: {train_data['p5'].dtype} | NaNs: {torch.isnan(train_data['p5']).any().item()}")
print(f"    - Labels: {list(train_data['labels'].shape)} | Classes: {torch.unique(train_data['labels']).tolist()}")

print(f"\n[+] VALIDATION TENSOR:")
print(f"    - p3: {list(val_data['p3'].shape)} | dtype: {val_data['p3'].dtype} | NaNs: {torch.isnan(val_data['p3']).any().item()}")
print(f"    - p4: {list(val_data['p4'].shape)} | dtype: {val_data['p4'].dtype} | NaNs: {torch.isnan(val_data['p4']).any().item()}")
print(f"    - p5: {list(val_data['p5'].shape)} | dtype: {val_data['p5'].dtype} | NaNs: {torch.isnan(val_data['p5']).any().item()}")
print(f"    - Labels: {list(val_data['labels'].shape)} | Classes: {torch.unique(val_data['labels']).tolist()}")

# Trực quan hóa phân bố nhãn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels_map = {0: "Tỉnh táo (Alert)", 1: "Buồn ngủ (Drowsy)"}
train_counts = [(train_data["labels"] == 0).sum().item(), (train_data["labels"] == 1).sum().item()]
val_counts = [(val_data["labels"] == 0).sum().item(), (val_data["labels"] == 1).sum().item()]

axes[0].bar(["Alert (0)", "Drowsy (1)"], train_counts, color=["steelblue", "darkorange"], width=0.5)
axes[0].set_title(f"Tập Train: Tổng {len(train_data['labels'])} mẫu", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Số lượng mẫu")
for i, v in enumerate(train_counts):
    axes[0].text(i, v + max(train_counts) * 0.02, str(v), ha="center", fontweight="bold")

axes[1].bar(["Alert (0)", "Drowsy (1)"], val_counts, color=["steelblue", "darkorange"], width=0.5)
axes[1].set_title(f"Tập Validation: Tổng {len(val_data['labels'])} mẫu", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Số lượng mẫu")
for i, v in enumerate(val_counts):
    axes[1].text(i, v + max(val_counts) * 0.02, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()
print("=" * 80)
print("[+] Tất cả dữ liệu hợp lệ và sẵn sàng cho Giai đoạn 2 (Huấn luyện Deep LSTM)!")
print("=" * 80)
